Vorige week heb je met behulp van Keras een neuraal netwerk gebouwd voor de MNIST‑dataset.


Dit netwerk gebruikte ingebouwde activatiefuncties zoals relu en softmax.



Deze week ga je:

- Activatiefuncties zelf implementeren
- Begrijpen wat pre‑activatie, logits en outputinterpretatie zijn
- Onderzoeken wat het effect van verschillende activatiefuncties is op het gedrag van je netwerk

### Pre‑activatie expliciet maken

Normaal voert Keras deze stap automatisch uit. In deze opdracht ga je dit zichtbaar maken.

#### Pre-Activation
Gegeven één neuron met:

inputs: x
gewichten: w
bias: b


Schrijf in Python een functie:

def ***pre_activation(x, w, b)***. Deze functie moet de pre‑activatie berekenen (dus nog zonder activatiefunctie).

Test je functie met zelfgekozen waarden.
Print de uitkomst en leg in commentaar uit:

wat deze waarde betekent
waarom dit nog geen output van het neuron is

In [1]:
import math
import numpy as np

In [2]:
def pre_activation(values, weights, bias):
    return sum([values[i] * weights[i] for i, _ in enumerate(weights) ]) + bias

print(pre_activation([1, 1], [2, 2], 1))
# pre-actiavtion = (1*2 + 1*2) + 1
# Dit is nog niet de output van de neuron omdat de activator functie nog niet is gebruikt.
# Deze zijn erg belangrijk omdat deze de neuron non linear maken (waarmee het beter op de data aansluit).

5


#### Activatiefuncties zelf maken

Tot nu toe heb je default activatiefunctie implementaties gebruikt, maat laten we nu kijken naar eigen implementatie om te begrijpen hoe het nou werkt. (Mogelijk heb je enkele van de functies al gemaakt bij de CodeGrade opdrachten van deze week.)

#### ReLU zelf implementeren

Schrijf een functie:

***def relu(z):***

De functie moet werken voor:

- 1 getal
- 1 numpy array


Test je functie met:

- negatieve waarden
- nul
- positieve waarden



#### Sigmoid zelf implementeren

Zoek de formule op. en implementeer de functie:

***def sigmoid(z):***

Test de functie met:

- grote negatieve waarden
- 0
- grote positieve waarden


- Wanneer gaat de output richting 0?
- Wanneer richting 1?
- Wat betekent een output van 0.5?


#### Softmax zelf implementeren

Zoek de formule op. en implementeer de functie:

Je krijgt de formule:

***def softmax(z):*** 

Test met verschillende lijstjes.

Let er op dat de outputs optellen tot 1.

waarom softmax ook al weer nuttig is bij multi‑class classificatie?

In [3]:
values = [-100, -10, -3, -2, -1, 0, 1, 2, 3, 10, 100]

def relu(value):
    return max(0, value)

def sigmoid(value):
    return 1 / (1 + math.exp(-value))

# Output van 0.5 ligt halverwegen tussen 0 en 1. Dit komt dus overeen met een input van 0.
# Output gaat richting 0 onder de 0.5
# Output gaat richting 1 boven 0.5

softmax_values_1 = [1, 0, -1]
softmax_values_2 = [0.1, 0.1, 0.1]

def softmax(values):
    return np.exp(values) / np.sum(np.exp(values))



print([relu(v) for v in values])
print([sigmoid(v) for v in values])

print(softmax(softmax_values_1))
print(sum(softmax(softmax_values_1)))
print(softmax(softmax_values_2))
print(sum(softmax(softmax_values_2)))


[0, 0, 0, 0, 0, 0, 1, 2, 3, 10, 100]
[3.7200759760208356e-44, 4.5397868702434395e-05, 0.04742587317756678, 0.11920292202211755, 0.2689414213699951, 0.5, 0.7310585786300049, 0.8807970779778823, 0.9525741268224334, 0.9999546021312976, 1.0]
[0.66524096 0.24472847 0.09003057]
1.0
[0.33333333 0.33333333 0.33333333]
1.0


#### Je eigen Activatiefuncties gebruiken in het netwerk

Maak een Keras‑model (of gebruik die uit de vorige practicum opdracht) waarin je geen activatiefunctie opgeeft in de layers
alleen Dense(units) gebruikt


- Wat zijn de outputs van de hidden layer nu?
- Wat stellen deze waarden conceptueel voor?

Gebruik nu dan je zelfgeschreven activatiefuncties om:

De output van de hidden layer handmatig door ReLU te halen
De output van de laatste layer door:

- sigmoid (single‑class experiment)
- softmax (multi‑class experiment)

Hoe kun je dit in je mnist db gebruiken(single‑class en multi-class)? Denk er over na.

Welke heeft het beste gewerkt? Waarom?

Probeer je netwerk zo te maken dat het nog steeds op de Mystery device past.



Antwoord:
- De outputs zijn willekeurige getallen
- Deze waardes zouden samenstellingen van eerdere lagen moeten voorstellen. Momenteel slagen ze er niet in verder te komen dan een willekeurige verdeling. Met 10% accuracy.

Deze zelf gedefineerde activator fucties hebben geen toegevoegde waarde. Ze zijn al ingebouwd. Dus zorgt de extra lambda layer alleen maar voor een groter model met meer overhead.

het lijkt er op dat er weinig verschil is tussen sigmoid en softmax in voorspellingskracht.


In [4]:
# Imports
import numpy as np;
from tensorflow.keras.datasets import mnist;
import tensorflow as tf;
import random as rnd;

# 60_000 training images
# 10_000 test images

(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

In [5]:
# Method Block

def OneHotEncodeDigit(label: int):
    """
    One hot encode a single one digit label
    """
    arr = np.zeros((10))
    arr[label] = 1
    return arr

def OneHotEncode(arr):
    """
    One hot encode an array of labels, like the train_images
    """
    return np.array([OneHotEncodeDigit(item) for item in arr])

def normalize_and_vectorize_image(image: np.array):
    """
    Turn a image of variable dimesions (must be square) into a vector and normalize values between 0 and 1
    """
    flat = image.flatten()

    maximum = np.max(flat)
    normalised: np.array = flat / maximum

    return normalised.astype(np.half)

def compres_image(image, new_size = 14):
    """
    Reduce the resolution of an image
    """
    originial_size = len(image)
    step = originial_size / new_size
    steps = [s * step for s in range(new_size)]
    new_image = np.zeros((new_size, new_size), dtype="uint8")
    y_i = 0
    for block_y_start in steps:
        block_y_end = block_y_start + step
        x_i = 0
        for block_x_start in steps:
            block_x_end = block_x_start + step
            block = image[int(block_y_start) : int(block_y_end), int(block_x_start): int(block_x_end)]
            pixel_color = np.average(block)
            new_image[y_i, x_i] = pixel_color
            x_i += 1
        y_i += 1
    return new_image


In [6]:
# ENCODE
test_labels_encoded = OneHotEncode(test_labels)
train_labels_encoded = OneHotEncode(train_labels)
test_images_normalised = np.array([normalize_and_vectorize_image(image) for image in test_images])
train_images_normalised = np.array([normalize_and_vectorize_image(image) for image in train_images])

In [7]:
#Shrink
test_images_normalised_small = np.array([normalize_and_vectorize_image(compres_image(image)) for image in test_images])
print("finished test imaged")
train_images_normalised_small = np.array([normalize_and_vectorize_image(compres_image(image)) for image in train_images])

finished test imaged


In [8]:
model = tf.keras.Sequential([
    tf.keras.Input(shape=(784,)),
    tf.keras.layers.Dense(140),
    tf.keras.layers.Dense(40),
    tf.keras.layers.Dense(10)
])

model.compile(optimizer='adam',
    loss='categorical_crossentropy',
    metrics=[
        "accuracy",
    ])

model.fit(train_images_normalised, train_labels_encoded, epochs=5, batch_size=128, verbose=1)

model.evaluate(test_images_normalised, test_labels_encoded)


Epoch 1/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.1456 - loss: 8.1304
Epoch 2/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.1166 - loss: 7.9963
Epoch 3/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.0812 - loss: 8.0786
Epoch 4/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.0602 - loss: 8.0208
Epoch 5/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.0552 - loss: 7.3866
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.0626 - loss: 6.6572


[6.657242298126221, 0.06260000169277191]

In [9]:
model.predict(np.array([test_images_normalised[1]]))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step


array([[  5.3246446 , -13.628087  ,  -3.9889467 , -23.088572  ,
          0.96800995,  24.756168  ,  24.718397  ,  11.37773   ,
        -14.275679  , -12.214597  ]], dtype=float32)

In [10]:
# Custom activation function
from keras.layers import Activation
from keras.utils import get_custom_objects

def custom_relu(x):
    return tf.maximum(0.0, x)

def custom_softmax(x):
    return tf.exp(x) / tf.reduce_sum(tf.exp(x))

def custom_sigmoid(x):
    return 1 / (1 + tf.exp(-x))

In [22]:
model = tf.keras.Sequential([
    tf.keras.Input(shape=(784,)),
    tf.keras.layers.Dense(40),
    tf.keras.layers.Lambda(custom_relu),
    tf.keras.layers.Dense(10),
    tf.keras.layers.Lambda(custom_softmax)
])


model.compile(optimizer='adam',
    loss='categorical_crossentropy',
    metrics=[
        "accuracy",
    ])



model.fit(train_images_normalised, train_labels_encoded, epochs=50, batch_size=128, verbose=1)


model.evaluate(test_images_normalised, test_labels_encoded)

Epoch 1/50
469/469 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.8717 - loss: 0.4691
Epoch 2/50
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9312 - loss: 0.2399
Epoch 3/50
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9448 - loss: 0.1926
Epoch 4/50
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9530 - loss: 0.1633
Epoch 5/50
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9584 - loss: 0.1429
Epoch 6/50
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9619 - loss: 0.1283
Epoch 7/50
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9657 - loss: 0.1157
Epoch 8/50
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9685 - loss: 0.1057
Epoch 9/50
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9707 - loss: 0.0976
Epoch 10/50
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9729 - loss: 0.0906
Epoch 11/50
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9748 - loss: 0.0839
Epoch 12/50
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step

[0.13537296652793884, 0.9681000113487244]

In [23]:
model = tf.keras.Sequential([
    tf.keras.Input(shape=(784,)),
    tf.keras.layers.Dense(40),
    tf.keras.layers.Lambda(custom_relu),
    tf.keras.layers.Dense(10),
    tf.keras.layers.Lambda(custom_sigmoid)
])


model.compile(optimizer='adam',
    loss='categorical_crossentropy',
    metrics=[
        "accuracy",
    ])



model.fit(train_images_normalised, train_labels_encoded, epochs=50, batch_size=128, verbose=1)


model.evaluate(test_images_normalised, test_labels_encoded)

Epoch 1/50
469/469 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.8603 - loss: 0.5245
Epoch 2/50
469/469 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.9332 - loss: 0.2388
Epoch 3/50
469/469 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9450 - loss: 0.1943
Epoch 4/50
469/469 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.9524 - loss: 0.1655
Epoch 5/50
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9582 - loss: 0.1454
Epoch 6/50
469/469 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.9618 - loss: 0.1318
Epoch 7/50
469/469 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.9651 - loss: 0.1211
Epoch 8/50
469/469 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.9675 - loss: 0.1119
Epoch 9/50
469/469 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.9698 - loss: 0.1048
Epoch 10/50
469/469 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.9709 - loss: 0.0973
Epoch 11/50
469/469 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.9729 - loss: 0.0915
Epoch 12/50
469/469 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/ste

[nan, 0.09799999743700027]

In [24]:
model.summary()

Model: "sequential_11"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_23 (Dense)                │ (None, 40)             │        31,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda_4 (Lambda)               │ (None, 40)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_24 (Dense)                │ (None, 10)             │           410 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda_5 (Lambda)               │ (None, 10)             │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 95,432 (372.79 KB)

 Trainable params: 31,810 (124.26 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 63,622 (248.53 KB)

In [25]:
import tracemalloc;
import time;

In [26]:


model_small = tf.keras.Sequential([
    tf.keras.Input(shape=(196,)),
    tf.keras.layers.Dense(3, 'relu'),
    tf.keras.layers.Dense(10, 'sigmoid')
])


model_small.compile(optimizer='adam',
    loss='categorical_crossentropy',
    metrics=[
        "accuracy",
    ])



model_small.fit(train_images_normalised_small, train_labels_encoded, epochs=2, batch_size=128, verbose=1)

time.sleep(1)

tracemalloc.start()
tracemalloc.clear_traces()

model_small.evaluate(test_images_normalised_small, test_labels_encoded)

curr, peak = tracemalloc.get_traced_memory()

print(f"curr: {curr/1000_000} mb | max {peak/1000_000} mb | differance {(peak-curr)/1000_000}")

model_small.summary()

Epoch 1/2
469/469 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.3769 - loss: 1.7656
Epoch 2/2
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6520 - loss: 1.1839
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7004 - loss: 1.0507
curr: 0.392396 mb | max 16.623641 mb | differance 16.231245


Model: "sequential_12"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_25 (Dense)                │ (None, 3)              │           591 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_26 (Dense)                │ (None, 10)             │            40 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,895 (7.41 KB)

 Trainable params: 631 (2.46 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 1,264 (4.94 KB)

Op basis van de hierbovenstaande gegevens is het goed mogelijk dat het model past op de mystery device. Maar zonder docker container/vm om deze na te boosten blijft het lastig hier met zekerheid uitsprken over te doen. tracemalloc geeft het beeld dat veel geheugen wordt verbruikt. Ik vermoed dat dit komt doordat er onnodige informatie wordt megerekend zoals alle mnist afbeeldingen of andere modellen. De summary van het model geeft een veel geringere grote weer van slechts 7.41 kb van alle parameters. Hier moet nog overhead bij worden gerekend waaronder de compressie functie en library imports.